# Word2Vec — CBOW and Skip-Gram Training

In this notebook we train two Word2Vec models from scratch using Gensim:
- **CBOW** (`sg=0`): predicts center word from context words
- **Skip-Gram** (`sg=1`): predicts context words from center word

We then display vocabulary size, word vectors, and compare top-5 similar words for both models.

In [1]:
# Install gensim if not already available
!pip install gensim -q

import re
from gensim.models import Word2Vec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 33.2 MB/s eta 0:00:00


## Step 1: Prepare the Text Dataset

We use a small manually written corpus. Each sentence will become one training sample.

In [2]:
# A small English corpus covering nature, royalty, and animals
raw_text = """
The cat sat on the mat. The cat is a small animal.
Dogs are loyal animals. Dogs love to play and run.
The king rules the kingdom. The queen rules beside the king.
A man walked into the forest. The woman followed the man.
The lion is the king of the jungle. The tiger is a fierce animal.
Birds fly high in the sky. Fish swim deep in the ocean.
The sun rises in the east and sets in the west.
Trees grow tall in the forest. Flowers bloom in spring.
The dog chased the cat around the house all day long.
Animals live in different habitats across the world.
The boy played with the dog in the park near the river.
The girl read a book about wild animals and their habitats.
Rivers flow from mountains to the ocean below.
The forest is home to many animals including deer and foxes.
Cats and dogs are the most popular pets in the world.
The lion hunts prey in the savanna grasslands of Africa.
Tigers are endangered animals that live in Asian forests.
Eagles are powerful birds that soar above mountain peaks.
Dolphins are intelligent animals that live in the ocean.
The wise old man told stories about the ancient kingdom.
A brave woman led her people through the dark forest safely.
Children love to learn about animals and nature around them.
The queen was as powerful as the king in the royal court.
"""

print(f"Total characters loaded: {len(raw_text)}")

Total characters loaded: 1300


## Step 2: Text Preprocessing

We lowercase the text, remove punctuation, and split into sentence-level token lists. Gensim expects a list of lists (each inner list = one sentence).

In [3]:
def preprocess(text):
    # Convert to lowercase so 'Cat' and 'cat' are treated as same word
    text = text.lower()

    # Remove everything except letters and spaces
    text = re.sub(r'[^a-z\s]', '', text)

    sentences = []
    for line in text.strip().split('\n'):
        words = line.strip().split()
        # Only keep lines with more than one word (skip empty lines)
        if len(words) > 1:
            sentences.append(words)
    return sentences

sentences = preprocess(raw_text)

# Flatten all sentences into one list to count total tokens
all_words = [word for sent in sentences for word in sent]

print(f"Total sentences : {len(sentences)}")
print(f"Total tokens    : {len(all_words)}")
print(f"Unique words    : {len(set(all_words))}")
print()
print("First 3 sentences after preprocessing:")
for s in sentences[:3]:
    print(' ', s)

Total sentences : 23
Total tokens    : 243
Unique words    : 130

First 3 sentences after preprocessing:
  ['the', 'cat', 'sat', 'on', 'the', 'mat', 'the', 'cat', 'is', 'a', 'small', 'animal']
  ['dogs', 'are', 'loyal', 'animals', 'dogs', 'love', 'to', 'play', 'and', 'run']
  ['the', 'king', 'rules', 'the', 'kingdom', 'the', 'queen', 'rules', 'beside', 'the', 'king']


## Step 3: Train CBOW and Skip-Gram Models

Both models use the same parameters. Only `sg` flag changes:
- `sg=0` → CBOW (faster, better for common words)
- `sg=1` → Skip-Gram (slower, better for rare words)

`epochs=200` is used because our dataset is very small.

In [4]:
# Train CBOW model (sg=0 means CBOW architecture)
cbow_model = Word2Vec(
    sentences=sentences,
    vector_size=50,   # each word gets a 50-dimensional vector
    window=3,         # look 3 words to left and right as context
    min_count=1,      # include all words even if they appear only once
    sg=0,             # 0 = CBOW
    epochs=200,       # train for 200 passes over the data
    seed=42           # fixed seed so results are reproducible
)

# Train Skip-Gram model (sg=1 means Skip-Gram architecture)
skipgram_model = Word2Vec(
    sentences=sentences,
    vector_size=50,
    window=3,
    min_count=1,
    sg=1,             # 1 = Skip-Gram
    epochs=200,
    seed=42
)

print("Both models trained successfully.")

Both models trained successfully.


## Step 4: Vocabulary Size

Since both models train on the same text, they share the same vocabulary.

In [5]:
# key_to_index gives us a dict of all known words
vocab = list(cbow_model.wv.key_to_index.keys())

print(f"Vocabulary size: {len(vocab)} unique words")
print()
# Sort and display all words alphabetically
print(sorted(vocab))

Vocabulary size: 130 unique words

['a', 'about', 'above', 'across', 'africa', 'all', 'ancient', 'and', 'animal', 'animals', 'are', 'around', 'as', 'asian', 'below', 'beside', 'birds', 'bloom', 'book', 'boy', 'brave', 'cat', 'cats', 'chased', 'children', 'court', 'dark', 'day', 'deep', 'deer', 'different', 'dog', 'dogs', 'dolphins', 'eagles', 'east', 'endangered', 'fierce', 'fish', 'flow', 'flowers', 'fly', 'followed', 'forest', 'forests', 'foxes', 'from', 'girl', 'grasslands', 'grow', 'habitats', 'her', 'high', 'home', 'house', 'hunts', 'in', 'including', 'intelligent', 'into', 'is', 'jungle', 'king', 'kingdom', 'learn', 'led', 'lion', 'live', 'long', 'love', 'loyal', 'man', 'many', 'mat', 'most', 'mountain', 'mountains', 'nature', 'near', 'ocean', 'of', 'old', 'on', 'park', 'peaks', 'people', 'pets', 'play', 'played', 'popular', 'powerful', 'prey', 'queen', 'read', 'rises', 'river', 'rivers', 'royal', 'rules', 'run', 'safely', 'sat', 'savanna', 'sets', 'sky', 'small', 'soar', 'spring

## Step 5: Word Vectors for 5 Words

Each word is stored as a vector of 50 floats. We display the first 10 values to keep the output readable.

In [6]:
# Five representative words from our corpus
words_to_show = ['cat', 'dog', 'king', 'woman', 'forest']

print(f"{'Word':<10}  {'Model':<12}  First 10 vector values")
print("-" * 70)

for word in words_to_show:
    for label, model in [("CBOW", cbow_model), ("Skip-Gram", skipgram_model)]:
        # model.wv[word] returns the full 50-dim vector for that word
        vec = model.wv[word]
        vals = ', '.join([f'{v:.3f}' for v in vec[:10]])
        print(f"{word:<10}  {label:<12}  [{vals}...]")
    print()  # blank line between words

Word        Model         First 10 vector values
----------------------------------------------------------------------
cat         CBOW          [-0.057, -0.038, -0.230, 0.314, -0.197, 0.303, -0.189, 0.405, -0.453, 0.012...]
cat         Skip-Gram     [-0.140, -0.067, -0.143, 0.362, -0.082, 0.134, -0.112, 0.185, -0.271, 0.097...]

dog         CBOW          [-0.024, -0.037, -0.174, 0.224, -0.196, 0.234, -0.181, 0.377, -0.355, -0.030...]
dog         Skip-Gram     [-0.085, -0.087, -0.113, 0.239, -0.185, 0.126, -0.192, 0.289, -0.233, 0.009...]

king        CBOW          [-0.034, -0.060, -0.187, 0.213, -0.150, 0.251, -0.178, 0.355, -0.379, -0.013...]
king        Skip-Gram     [-0.128, -0.123, -0.154, 0.160, -0.080, 0.148, -0.192, 0.255, -0.293, -0.050...]

woman       CBOW          [-0.011, -0.034, -0.200, 0.260, -0.167, 0.260, -0.199, 0.368, -0.398, -0.020...]
woman       Skip-Gram     [-0.082, -0.078, -0.204, 0.253, 0.001, 0.115, -0.217, 0.272, -0.314, -0.074...]

forest      CBOW        

## Step 6: Top-5 Similar Words — CBOW vs Skip-Gram

Cosine similarity is used to find the most similar words in the vector space. Both models may agree or disagree based on how they learned context.

In [7]:
# Query words for similarity comparison
query_words = ['cat', 'king', 'forest', 'animal', 'man']

for qw in query_words:
    print(f"Query word: '{qw}'")
    print(f"  {'Rank':<5}  {'CBOW':<30}  {'Skip-Gram'}")
    print(f"  {'----':<5}  {'-'*28:<30}  {'-'*28}")

    # Get top 5 most similar words from each model
    cbow_results  = cbow_model.wv.most_similar(qw, topn=5)
    sg_results    = skipgram_model.wv.most_similar(qw, topn=5)

    for i, ((cw, cs), (sw, ss)) in enumerate(zip(cbow_results, sg_results), 1):
        cb_str = f"{cw:<15} {cs:.4f}"
        sg_str = f"{sw:<15} {ss:.4f}"
        print(f"  {i:<5}  {cb_str:<30}  {sg_str}")
    print()

Query word: 'cat'
  Rank   CBOW                            Skip-Gram
  ----   ----------------------------    ----------------------------
  1      day             0.9965          mat             0.9867
  2      mat             0.9962          on              0.9859
  3      the             0.9959          sat             0.9836
  4      a               0.9959          small           0.9783
  5      on              0.9959          house           0.9601

Query word: 'king'
  Rank   CBOW                            Skip-Gram
  ----   ----------------------------    ----------------------------
  1      the             0.9962          beside          0.9820
  2      is              0.9954          rules           0.9818
  3      a               0.9952          queen           0.9813
  4      of              0.9949          the             0.9693
  5      dog             0.9949          was             0.9664

Query word: 'forest'
  Rank   CBOW                            Skip-Gram
  ---- 

## Conclusion

- **CBOW** averages context word vectors to predict the center word. It is faster and works well when the corpus is small or words are frequent.
- **Skip-Gram** uses the center word to predict each surrounding context word separately. It generalizes better to rare words.
- Both models encode word meaning geometrically — similar words end up with similar vectors.
- On a larger corpus, the famous analogy `king - man + woman ≈ queen` would work via simple vector arithmetic.